# RF-DETR Infrastructure Detection Training

**BAHB Project - Power Infrastructure Detection**

This notebook trains RF-DETR on the BAHB infrastructure detection dataset.

## Requirements
- Google Colab with GPU (T4/A100 recommended)
- ~15GB disk space for dataset
- ~4-6 hours training time

## Dataset Stats
- 13,774 training images (168,413 annotations)
- 4,909 validation images (64,035 annotations)
- 10 infrastructure classes with data

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repository
!git clone https://github.com/abdillahia-lab/BAHB.git
%cd BAHB

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers pycocotools tqdm pillow opencv-python-headless scipy

In [ ]:
# Upload dataset (if not in repo)
# Option 1: Upload from Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy dataset from Drive (adjust path as needed)
# !cp -r '/content/drive/MyDrive/BAHB/data/rf_detr' data/

In [ ]:
# Option 2: Convert dataset if YOLO format exists
!python scripts/yolo_to_coco_converter.py --input-dir data/merged --output-dir data/rf_detr --no-masks

In [ ]:
# Verify dataset
import json
with open('data/rf_detr/metadata.json') as f:
    meta = json.load(f)
    
print(f"Training: {meta['training_set']['images']} images, {meta['training_set']['annotations']} annotations")
print(f"Validation: {meta['validation_set']['images']} images, {meta['validation_set']['annotations']} annotations")
print(f"Classes with data: {17 - len(meta['missing_classes'])}")
print(f"Missing classes: {[c['name'] for c in meta['missing_classes']]}")

In [ ]:
# Training Configuration
MODEL_SIZE = 'medium'  # Options: nano, small, medium, base, large
EPOCHS = 100
BATCH_SIZE = 8  # Reduce to 4 for T4, increase to 16 for A100
LEARNING_RATE = 1e-4

In [ ]:
# Start Training
!python train_rf_detr_seg.py \
    --data data/rf_detr \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --model-size {MODEL_SIZE} \
    --output runs/rf_detr_{MODEL_SIZE}

In [ ]:
# Export to ONNX after training
!python train_rf_detr_seg.py \
    --export \
    --checkpoint runs/rf_detr_{MODEL_SIZE}/best.pt \
    --output models/onnx/rf_detr_{MODEL_SIZE}.onnx

In [ ]:
# Download trained model
from google.colab import files
files.download(f'runs/rf_detr_{MODEL_SIZE}/best.pt')

In [ ]:
# Save to Google Drive
!cp -r runs/rf_detr_{MODEL_SIZE} '/content/drive/MyDrive/BAHB/runs/'